In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
import sys

In [3]:
from pathlib import Path
project_path = Path("/workspace")
api_folder = project_path / "08_api"
api_folder.mkdir(parents=True, exist_ok=True)
print("Folder created:", api_folder)
print("Folder exists:", api_folder.exists())

Folder created: /workspace/08_api
Folder exists: True


In [10]:
from pathlib import Path

api_path = Path("/workspace/08_api")

# Create application folder
app_path = api_path / "app"
app_path.mkdir(parents=True, exist_ok=True)

files = [
    app_path / "__init__.py",
    app_path / "main.py",
    api_path / "README.md"
]

for file in files:
    file.touch(exist_ok=True)

print("API structure created:\n")

for item in sorted(api_path.rglob("*")):
    print(item.relative_to(api_path))

API structure created:

README.md
app
app/__init__.py
app/main.py


In [11]:
from pathlib import Path

project_path = Path("/workspace")

matches = list(project_path.rglob("main.py"))

if matches:
    print("Found main.py files:")
    for path in matches:
        print(path)
else:
    print("No main.py files found inside /workspace.")

Found main.py files:
/workspace/08_api/app/main.py


In [13]:
# checking task 5-7 

In [14]:
project_path = Path("/workspace")

folders = ["models", "src", "tests"]

for folder in folders:
    folder_path = project_path / folder

    print(f"\n{'='*40}")
    print(f"{folder}/")
    print(f"{'='*40}")

    if folder_path.exists():
        items = list(folder_path.rglob("*"))

        if items:
            for item in items:
                if item.is_file():
                    print(item.relative_to(project_path))
        else:
            print("Folder is empty.")
    else:
        print("Folder does not exist.")


models/
models/attrition_preprocessor.joblib
models/metadata.json

src/
src/feature.py
src/features.py
src/.ipynb_checkpoints/feature-checkpoint.py
src/.ipynb_checkpoints/features-checkpoint.py
src/__pycache__/feature.cpython-312.pyc
src/__pycache__/features.cpython-312.pyc

tests/
Folder is empty.


In [ ]:
# load both the exported model

In [17]:
import joblib
from pathlib import Path
from sklearn.utils.validation import check_is_fitted

models_path = Path("/workspace/models")

# Load attendance model
attendance_artifact = joblib.load(
    models_path / "attendance_detector.joblib"
)

# Load payroll model
payroll_artifact = joblib.load(
    models_path / "payroll_detector.joblib"
)

# Verify fitted models
check_is_fitted(attendance_artifact["model"])
check_is_fitted(payroll_artifact["model"])
check_is_fitted(payroll_artifact["imputer"])

print("Attendance model: Loaded and fitted")
print("Payroll model: Loaded and fitted")
print("\nAttendance features:",
      len(attendance_artifact["feature_columns"]))
print("Payroll features:",
      payroll_artifact["feature_columns"])

Attendance model: Loaded and fitted
Payroll model: Loaded and fitted

Attendance features: 16
Payroll features: ['gross_fixed_ratio', 'deduction_ratio', 'net_ratio']


In [18]:
file_path = Path(
    "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx"
)

sample_results = pd.read_excel(
    file_path,
    sheet_name="sample_anomaly_results"
)

attendance_sample = sample_results[
    sample_results["source_dataset"] == "attendance_monthly_features"
].copy()

print("Attendance sample records:", len(attendance_sample))

print("\nAttendance reason codes and explanations:")

display(
    attendance_sample[
        ["reason_code", "reason_text"]
    ]
    .drop_duplicates()
    .sort_values("reason_code")
    .reset_index(drop=True)
)

Attendance sample records: 221

Attendance reason codes and explanations:


,reason_code,reason_text
0,FREQUENT_LATE,15 late arrivals this month
1,FREQUENT_LATE,14 late arrivals this month
2,FREQUENT_LATE,19 late arrivals this month
3,LONG_UNPAID_ABSENCE,13 consecutive unpaid absence days
4,LONG_UNPAID_ABSENCE,14 consecutive unpaid absence days
5,LONG_UNPAID_ABSENCE,11 consecutive unpaid absence days
6,LONG_UNPAID_ABSENCE,12 consecutive unpaid absence days
7,LONG_UNPAID_ABSENCE,10 consecutive unpaid absence days
8,LONG_UNPAID_ABSENCE,9 consecutive unpaid absence days
9,LONG_UNPAID_ABSENCE,8 consecutive unpaid absence days


In [19]:
import json
import pandas as pd
from pathlib import Path

project_path = Path("/workspace")

# Load attendance data
attendance_df = pd.read_excel(
    project_path / "data/raw/02_CortexHR_AI_ML_Dataset.xlsx",
    sheet_name="attendance_monthly_features"
)

print("Attendance dataset shape:", attendance_df.shape)

print("\nAttendance columns:")
print(attendance_df.columns.tolist())


# Inspect existing Task 5 logic
notebook_path = project_path / "05_attendance_anomaly.ipynb"
with open(notebook_path, "r", encoding="utf-8") as file:
    notebook = json.load(file)
keywords = [
    "late", "absence", "zero attendance",
    "attendance_rate", "presence", "rule",
    "peer", "anomaly_flag"
]
print("\n" + "=" * 60)
print("RELEVANT TASK 5 CODE")
print("=" * 60)
for i, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] != "code":
        continue
    source = "".join(cell["source"])
    if any(word in source.lower() for word in keywords):
        print(f"\n--- Cell {i} ---")
        print(source[:3000])

Attendance dataset shape: (7339, 16)

Attendance columns:
['attendance_feature_id', 'employee_id', 'month', 'department_id', 'job_level', 'scheduled_days', 'present_days', 'wfh_days', 'half_days_unpaid', 'paid_leave_days', 'unpaid_absence_days', 'longest_unpaid_streak_days', 'late_arrivals', 'avg_hours_worked', 'overtime_hours', 'attendance_rate']

RELEVANT TASK 5 CODE

--- Cell 7 ---
model = IsolationForest(
    contamination=0.03,
    random_state=42
)
predictions = model.fit_predict(attendance_features)
attendance_df["anomaly_flag"] = (predictions == -1).astype(int)

attendance_df["anomaly_score"] = -model.decision_function(
    attendance_features
)

print("\nTotal records:", len(attendance_df))
print("Anomalies flagged:", attendance_df["anomaly_flag"].sum())
print("Anomaly percentage:",
      round(attendance_df["anomaly_flag"].mean() * 100, 2), "%")

--- Cell 8 ---
# attendance rule

--- Cell 9 ---
attendance_df["zero_attendance"] = (
    attendance_df["attendance_rate"] == 0
)
a

In [20]:
import pandas as pd
from pathlib import Path

file_path = Path(
    "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx"
)

sample_results = pd.read_excel(
    file_path,
    sheet_name="sample_anomaly_results"
)

attendance_data = pd.read_excel(
    file_path,
    sheet_name="attendance_monthly_features"
)

# Select attendance sample results
attendance_sample = sample_results[
    sample_results["source_dataset"] == "attendance_monthly_features"
].copy()

# Match sample results with original attendance features
attendance_review = attendance_sample.merge(
    attendance_data,
    left_on="record_id",
    right_on="attendance_feature_id",
    how="left"
)

print("Matched records:",
      attendance_review["attendance_feature_id"].notna().sum(),
      "out of", len(attendance_review))

print("\nReason code counts:")
display(attendance_review["reason_code"].value_counts())

# Features that may explain each attendance reason
check_columns = [
    "late_arrivals",
    "unpaid_absence_days",
    "longest_unpaid_streak_days",
    "attendance_rate",
    "present_days",
    "scheduled_days"
]

print("\nFeature statistics by reason code:")

display(
    attendance_review.groupby("reason_code")[check_columns]
    .agg(["min", "median", "max"])
    .round(2)
)

print("\nExamples of frequent late records:")
display(
    attendance_review[
        attendance_review["reason_code"] == "FREQUENT_LATE"
    ][["record_id", "late_arrivals", "reason_text"]].head(10)
)

print("\nExamples of long unpaid absence records:")
display(
    attendance_review[
        attendance_review["reason_code"] == "LONG_UNPAID_ABSENCE"
    ][[
        "record_id",
        "unpaid_absence_days",
        "longest_unpaid_streak_days",
        "reason_text"
    ]].head(10)
)

Matched records: 221 out of 221

Reason code counts:


reason_code
PATTERN_SHIFT          125
NO_ATTENDANCE           50
LONG_UNPAID_ABSENCE     43
FREQUENT_LATE            3
Name: count, dtype: int64


Feature statistics by reason code:


late_arrivals            unpaid_absence_days             \
                              min median max                 min median max   
reason_code                                                                   
FREQUENT_LATE                  14   15.0  19                   0    0.0   0   
LONG_UNPAID_ABSENCE             0    1.0   6                   7   10.0  14   
NO_ATTENDANCE                   0    0.0   0                   0    0.0  22   
PATTERN_SHIFT                   0    1.0  11                   0    0.0   7   

                    longest_unpaid_streak_days            attendance_rate  \
                                           min median max             min   
reason_code                                                                 
FREQUENT_LATE                                0    0.0   0            0.70   
LONG_UNPAID_ABSENCE                          7   10.0  14            0.21   
NO_ATTENDANCE                                0    0.0  22            0.00   
PATTERN_SHIFT                                0    0.0   6            0.04   

                                 present_days            scheduled_days  \
                    median   max          min median max            min   
reason_code                                                               
FREQUENT_LATE         0.79  0.95           14   15.0  19             19   
LONG_UNPAID_ABSENCE   0.47  0.65            4   10.0  14             18   
NO_ATTENDANCE         0.00  0.00            0    0.0   0              1   
PATTERN_SHIFT         0.65  1.00            1   11.0  21              1   

                                
                    median max  
reason_code                     
FREQUENT_LATE         20.0  20  
LONG_UNPAID_ABSENCE   21.0  23  
NO_ATTENDANCE         20.0  23  
PATTERN_SHIFT         20.0  23


Examples of frequent late records:


,record_id,late_arrivals,reason_text
165,AF-00532,15,15 late arrivals this month
186,AF-04013,14,14 late arrivals this month
207,AF-04614,19,19 late arrivals this month



Examples of long unpaid absence records:


,record_id,unpaid_absence_days,longest_unpaid_streak_days,reason_text
15,AF-03098,13,13,13 consecutive unpaid absence days
16,AF-04681,13,13,13 consecutive unpaid absence days
17,AF-01620,14,14,14 consecutive unpaid absence days
18,AF-01480,11,11,11 consecutive unpaid absence days
19,AF-02473,13,13,13 consecutive unpaid absence days
21,AF-06074,14,14,14 consecutive unpaid absence days
22,AF-02205,12,12,12 consecutive unpaid absence days
23,AF-04604,10,10,10 consecutive unpaid absence days
24,AF-04182,14,14,14 consecutive unpaid absence days
25,AF-02143,13,13,13 consecutive unpaid absence days


In [21]:
import pandas as pd
from pathlib import Path

file_path = Path(
    "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx"
)

sample_results = pd.read_excel(
    file_path,
    sheet_name="sample_anomaly_results"
)

attendance_data = pd.read_excel(
    file_path,
    sheet_name="attendance_monthly_features"
)

attendance_review = sample_results[
    sample_results["source_dataset"] == "attendance_monthly_features"
].merge(
    attendance_data,
    left_on="record_id",
    right_on="attendance_feature_id",
    how="left"
)

def assign_attendance_reason(row):

    if row["attendance_rate"] == 0:
        return "NO_ATTENDANCE"

    elif row["longest_unpaid_streak_days"] >= 7:
        return "LONG_UNPAID_ABSENCE"

    elif row["late_arrivals"] >= 14:
        return "FREQUENT_LATE"

    else:
        return "PATTERN_SHIFT"


attendance_review["candidate_reason_code"] = (
    attendance_review.apply(
        assign_attendance_reason,
        axis=1
    )
)

# Compare candidate rules with official reason codes
attendance_review["reason_match"] = (
    attendance_review["reason_code"]
    == attendance_review["candidate_reason_code"]
)

print("Total sample records:", len(attendance_review))
print("Correct classifications:",
      attendance_review["reason_match"].sum())
print("Incorrect classifications:",
      (~attendance_review["reason_match"]).sum())

print("\nCandidate reason-code counts:")
display(
    attendance_review["candidate_reason_code"].value_counts()
)

print("\nOfficial vs candidate comparison:")
display(
    pd.crosstab(
        attendance_review["reason_code"],
        attendance_review["candidate_reason_code"],
        rownames=["Official"],
        colnames=["Candidate"]
    )
)

print("\nMismatched records:")
display(
    attendance_review.loc[
        ~attendance_review["reason_match"],
        [
            "record_id",
            "reason_code",
            "candidate_reason_code",
            "attendance_rate",
            "longest_unpaid_streak_days",
            "late_arrivals"
        ]
    ]
)

print("\nOfficial reason text examples:")
for code, group in attendance_review.groupby("reason_code"):
    print(f"\n{code}:")
    for text_value in group["reason_text"].drop_duplicates().head(3):
        print("-", text_value)

Total sample records: 221
Correct classifications: 221
Incorrect classifications: 0

Candidate reason-code counts:


candidate_reason_code
PATTERN_SHIFT          125
NO_ATTENDANCE           50
LONG_UNPAID_ABSENCE     43
FREQUENT_LATE            3
Name: count, dtype: int64


Official vs candidate comparison:


Candidate,FREQUENT_LATE,LONG_UNPAID_ABSENCE,NO_ATTENDANCE,PATTERN_SHIFT
Official,,,,
FREQUENT_LATE,3,0,0,0
LONG_UNPAID_ABSENCE,0,43,0,0
NO_ATTENDANCE,0,0,50,0
PATTERN_SHIFT,0,0,0,125



Mismatched records:


,record_id,reason_code,candidate_reason_code,attendance_rate,longest_unpaid_streak_days,late_arrivals



Official reason text examples:

FREQUENT_LATE:
- 15 late arrivals this month
- 14 late arrivals this month
- 19 late arrivals this month

LONG_UNPAID_ABSENCE:
- 13 consecutive unpaid absence days
- 14 consecutive unpaid absence days
- 11 consecutive unpaid absence days

NO_ATTENDANCE:
- No presence recorded for the whole month

PATTERN_SHIFT:
- Attendance pattern differs sharply from peers and own history


In [24]:
#checking the API dependencies 

In [25]:
from importlib.util import find_spec


In [26]:
packages = [
    "fastapi",
    "pydantic",
    "uvicorn",
    "pytest",
    "httpx",
    "joblib",
    "sklearn",
    "pandas"
]

print("PACKAGE INSTALLATION STATUS\n")

for package in packages:
    status = "Installed" if find_spec(package) else "Not installed"
    print(f"{package}: {status}")
    
requirements_path = Path("/workspace/requirements.txt")

print("\nEXISTING REQUIREMENTS.TXT\n")

if requirements_path.exists():
    print(requirements_path.read_text())
else:
    print("requirements.txt not found")

PACKAGE INSTALLATION STATUS

fastapi: Not installed
pydantic: Not installed
uvicorn: Not installed
pytest: Not installed
httpx: Installed
joblib: Installed
sklearn: Installed
pandas: Installed

EXISTING REQUIREMENTS.TXT

﻿pandas
numpy
matplotlib
seaborn
jupyterlab
openpyxl
scikit-learn
scipy
xlrd
ipykernel


In [27]:
#install the API dependencies

In [30]:
import subprocess
requirements_path = Path("/workspace/requirements.txt")
existing = requirements_path.read_text(
    encoding="utf-8-sig"
).splitlines()
new_packages = [
    "fastapi",
    "pydantic",
    "uvicorn",
    "pytest",
    "httpx"
]
for package in new_packages:
    if package.lower() not in [
        line.strip().lower() for line in existing
    ]:
        existing.append(package)
requirements_path.write_text(
    "\n".join(existing).rstrip() + "\n",
    encoding="utf-8"
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "fastapi",
        "pydantic",
        "uvicorn",
        "pytest",
        "httpx"
    ],
    check=True
)
print("\nInstallation completed!")

requirements.txt updated!

Installation completed!



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [29]:
from importlib.metadata import version

packages = [
    "fastapi",
    "pydantic",
    "uvicorn",
    "pytest",
    "httpx"
]

for package in packages:
    print(f"{package}: {version(package)}")

fastapi: 0.162.0
pydantic: 2.14.2
uvicorn: 0.54.0
pytest: 9.2.0
httpx: 0.28.1


In [32]:
import joblib
project_path = Path("/workspace")
models_path = project_path / "models"
attendance_artifact = joblib.load(
    models_path / "attendance_detector.joblib"
)
payroll_artifact = joblib.load(
    models_path / "payroll_detector.joblib"
)
print("=" * 55)
print("ATTENDANCE API INPUT FEATURES")
print("=" * 55)

for feature in attendance_artifact["feature_columns"]:
    print(feature)

print("\nTotal attendance features:",
      len(attendance_artifact["feature_columns"]))
print("\n" + "=" * 55)
print("PAYROLL API INPUT FEATURES")
print("=" * 55)

for feature in payroll_artifact["feature_columns"]:
    print(feature)

excel_path = (
    project_path / "data/raw/02_CortexHR_AI_ML_Dataset.xlsx"
)

attendance_df = pd.read_excel(
    excel_path,
    sheet_name="attendance_monthly_features"
)

payroll_df = pd.read_excel(
    excel_path,
    sheet_name="payroll_monthly_features"
)

print("\nATTENDANCE DATA TYPES")
print(attendance_df.dtypes.to_string())

print("\nPAYROLL DATA TYPES")
print(payroll_df.dtypes.to_string())

ATTENDANCE API INPUT FEATURES
job_level
scheduled_days
present_days
wfh_days
half_days_unpaid
paid_leave_days
unpaid_absence_days
longest_unpaid_streak_days
late_arrivals
avg_hours_worked
overtime_hours
attendance_rate
anomaly_flag
anomaly_score
combined_anomaly_flag
true_anomaly

Total attendance features: 16

PAYROLL API INPUT FEATURES
gross_fixed_ratio
deduction_ratio
net_ratio

ATTENDANCE DATA TYPES
attendance_feature_id                    str
employee_id                              str
month                         datetime64[us]
department_id                            str
job_level                              int64
scheduled_days                         int64
present_days                           int64
wfh_days                               int64
half_days_unpaid                       int64
paid_leave_days                        int64
unpaid_absence_days                    int64
longest_unpaid_streak_days             int64
late_arrivals                          int64
avg_hour

In [34]:
models_path = Path("/workspace/models")

artifact = joblib.load(
    models_path / "attendance_detector.joblib"
)

model = artifact["model"]

print("Saved artifact feature count:",
      len(artifact["feature_columns"]))

print("Model expected feature count:",
      model.n_features_in_)

print("\nFeatures stored in artifact:")
print(artifact["feature_columns"])

print("\nFeatures recorded by fitted model:")
print(model.feature_names_in_.tolist()
      if hasattr(model, "feature_names_in_")
      else "Feature names unavailable")

print("\nSaved feature names match fitted model:",
      list(artifact["feature_columns"])
      == list(model.feature_names_in_)
      if hasattr(model, "feature_names_in_")
      else "Cannot verify")

Saved artifact feature count: 12
Model expected feature count: 12

Features stored in artifact:
['job_level', 'scheduled_days', 'present_days', 'wfh_days', 'half_days_unpaid', 'paid_leave_days', 'unpaid_absence_days', 'longest_unpaid_streak_days', 'late_arrivals', 'avg_hours_worked', 'overtime_hours', 'attendance_rate']

Features recorded by fitted model:
['job_level', 'scheduled_days', 'present_days', 'wfh_days', 'half_days_unpaid', 'paid_leave_days', 'unpaid_absence_days', 'longest_unpaid_streak_days', 'late_arrivals', 'avg_hours_worked', 'overtime_hours', 'attendance_rate']

Saved feature names match fitted model: True


In [35]:
#Create the API request and response schemas 

In [36]:
app_path = Path("/workspace/08_api/app")

In [37]:
app_path.mkdir(parents=True, exist_ok=True)

In [154]:
schema_code=''' 
from datetime import date,datetime
from typing import Literal
from pydantic import BaseModel,ConfigDict,Field 

# Attedance request 
class AttendanceRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    attendance_feature_id: str
    employee_id: str
    month: date
    job_level: int
    scheduled_days: int
    present_days: int
    wfh_days: int
    half_days_unpaid: int
    paid_leave_days: int
    unpaid_absence_days: int
    longest_unpaid_streak_days: int
    late_arrivals: int
    avg_hours_worked: float
    overtime_hours: float
    attendance_rate: float

#Payroll request
class PayrollRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    payroll_item_id: str
    employee_id: str
    run_id: str
    month: date
    monthly_fixed_gross: float
    payable_days: float
    gross_pay: float
    total_deductions: float
    net_pay: float

#Common API response
class AnomalyResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")
    result_id: str
    source_dataset: Literal[
        "attendance_monthly_features","payroll_monthly_features"]
    record_id: str
    employee_id: str
    period: date
    anomaly_score: float = Field(ge=0, le=1)
    flagged: Literal["Y", "N"]
    reason_code: str
    reason_text: str
    money_at_risk: float | None = Field(default=None, ge=0)
    model_version: str
    scored_at: datetime
'''
schema_file=app_path/"schemas.py"
schema_file.write_text(schema_code.strip()+"\n",encoding="utf-8")
print("API schema created successfully")
print("FIle",schema_file)
schema_file.exists()

API schema created successfully
FIle /workspace/08_api/app/schemas.py


True

In [60]:
#Testing the API schemas 

In [63]:
import importlib
api_path="/workspace/08_api"
if api_path not in sys.path:
    sys.path.insert(0,api_path)
from app.schemas import ( AttendanceRequest,PayrollRequest,AnomalyResponse)
from pydantic import ValidationError
print("All API schemas imported successfully")

All API schemas imported successfully


In [64]:
# 1. Test attendance request

In [80]:
attendance_df = pd.read_excel("/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx",sheet_name="attendance_monthly_features")
attendance_fields = list(AttendanceRequest.model_fields)
attendance_row = attendance_df.dropna(subset=attendance_fields).iloc[0]
attendance_payload = {
    field: attendance_row[field]
    for field in attendance_fields
}
attendance_request = AttendanceRequest(**attendance_payload)
print("\n Attendance request:VALID")
print("Employee ID:", attendance_request.employee_id)


 Attendance request:VALID
Employee ID: EMP-0002


In [81]:
# 2. Test payroll request

In [82]:
payroll_df = pd.read_excel("/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx",sheet_name="payroll_monthly_features")
payroll_fields = list(PayrollRequest.model_fields)
payroll_row = payroll_df.dropna(subset=payroll_fields).iloc[0]
payroll_payload = {
    field: payroll_row[field]
    for field in payroll_fields
}
payroll_request = PayrollRequest(**payroll_payload)
print("\nPayroll request: VALID")
print("Payroll item ID:", payroll_request.payroll_item_id)


Payroll request: VALID
Payroll item ID: PI-000001


In [83]:
# 3. Test invalid attendance input

In [84]:
invalid_payload=attendance_payload.copy()
invalid_payload.pop("attendance_rate")

try:
    AttendanceRequest(**invalid_payload)
    print("\nError:Invalid request was accepted")
except ValidationError:
    print("\nInvalid attendance request: REJECTED")
    print("Pydantic validation is working!")


Invalid attendance request: REJECTED
Pydantic validation is working!


In [85]:
# 4. check response fields 

In [86]:
print("\nOfficial response fields:")
print(list(AnomalyResponse.model_fields))


Official response fields:
['result_id', 'source_dataset', 'record_id', 'employee_id', 'period', 'anomaly_score', 'flagged', 'reason_code', 'reason_text', 'money_at_risk', 'model_version', 'scored_at']


In [ ]:
# verfying all artifacts 

In [93]:
models_path = Path("/workspace/models")
attendance = joblib.load(models_path / "attendance_detector.joblib")
payroll = joblib.load(models_path / "payroll_detector.joblib")
reason_codes = joblib.load(models_path / "reason_codes.joblib")
check_is_fitted(attendance["model"])
assert len(attendance["feature_columns"]) == 12
assert attendance["score_min"] < attendance["score_max"]
check_is_fitted(payroll["model"])
check_is_fitted(payroll["imputer"])
assert len(payroll["feature_columns"]) == 3
assert payroll["score_min"] < payroll["score_max"]
                                                                                 # Verify reason code configuration
assert "attendance" in reason_codes
assert "payroll" in reason_codes

print("all artifacts verified successfully")
print("\nAttendance:")
print("Features:", len(attendance["feature_columns"]))
print("Score range:", attendance["score_min"], "to", attendance["score_max"])
print("\nPayroll:")
print("Features:", len(payroll["feature_columns"]))
print("Imputer: Fitted")
print("Score range:", payroll["score_min"], "to", payroll["score_max"])
print("\nReason codes:")
print("Attendance rules:", len(reason_codes["attendance"]["rules"]))
print("Payroll definitions:", len(reason_codes["payroll"]["reason_definitions"]))

all artifacts verified successfully

Attendance:
Features: 12
Score range: -0.2212917370732435 to 0.18116446612782788

Payroll:
Features: 3
Imputer: Fitted
Score range: -0.19236282643250202 to 0.2685678649617652

Reason codes:
Attendance rules: 4
Payroll definitions: 4


In [91]:
#load all the models 

In [155]:

import py_compile
main_code='''
import joblib
from fastapi import FastAPI, Request
from contextlib import asynccontextmanager
from pathlib import Path
PROJECT_ROOT = Path(__file__).resolve().parents[2]
MODELS_DIR = PROJECT_ROOT / "models"
#load all artifacts once during application startup
@asynccontextmanager
async def lifespan(app: FastAPI):
    attendance_path = MODELS_DIR / "attendance_detector.joblib"
    payroll_path = MODELS_DIR / "payroll_detector.joblib"
    reason_codes_path = MODELS_DIR / "reason_codes.joblib"
    required_files = [
        attendance_path,
        payroll_path,
        reason_codes_path
    ]
    for file in required_files:
        if not file.exists():
            raise FileNotFoundError(
                f"Required model artifact not found: {file}"
            )
    app.state.artifacts = {
        "attendance": joblib.load(attendance_path),
        "payroll": joblib.load(payroll_path),
        "reason_codes": joblib.load(reason_codes_path)
    }
    print("All anomaly service artifacts loaded successfully.")
    yield
    app.state.artifacts = {}

# Create FastAPI application
app = FastAPI(
    title="CortexHR Anomaly Service",
    description="Attendance and payroll anomaly detection API",
    version="1.0.0",
    lifespan=lifespan
)
# Health-check endpoint
@app.get("/health")
def health_check(request: Request):
    return {
        "status": "healthy",
        "loaded_artifacts": list(
            request.app.state.artifacts.keys()
        )
    }
'''
main_path = Path("/workspace/08_api/app/main.py")
main_path.write_text(main_code.strip() + "\n",encoding="utf-8")

1472

In [156]:
print("main.py created successfully!")
print("Location:", main_path)
main_path.exists()

main.py created successfully!
Location: /workspace/08_api/app/main.py


True

In [157]:
# test model loading at stratup 

In [158]:
import sys
import importlib

api_path = "/workspace/08_api"

if api_path not in sys.path:
    sys.path.insert(0, api_path)

# Reload application module
for name in list(sys.modules):
    if name == "app.main":
        del sys.modules[name]

from fastapi.testclient import TestClient
from app.main import app

with TestClient(app) as client:
    response = client.get("/health")

    print("HTTP Status:", response.status_code)
    print("Response:", response.json())

    assert response.status_code == 200

print("\nStartup verification successful!")

All anomaly service artifacts loaded successfully.
HTTP Status: 200
Response: {'status': 'healthy', 'loaded_artifacts': ['attendance', 'payroll', 'reason_codes']}

Startup verification successful!


In [159]:
#attedance_endpoint

In [160]:
file_path = "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx"
sample_results = pd.read_excel(
    file_path,
    sheet_name="sample_anomaly_results"
)
attendance_sample = sample_results[ sample_results["source_dataset"] == "attendance_monthly_features"].copy()
print("Attendance sample output:")

attendance_sample[
        [
            "result_id",
            "record_id",
            "employee_id",
            "period",
            "anomaly_score",
            "flagged",
            "reason_code",
            "reason_text",
            "money_at_risk",
            "model_version",
            "scored_at"
        ]
    ].head(5)


Attendance sample output:


,result_id,record_id,employee_id,period,anomaly_score,flagged,reason_code,reason_text,money_at_risk,model_version,scored_at
0,AN-0001,AF-03170,EMP-0356,2025-07-01,1.000,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00
1,AN-0002,AF-05918,EMP-0585,2025-09-01,0.998,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00
2,AN-0003,AF-06802,EMP-0646,2026-06-01,0.998,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00
3,AN-0004,AF-02819,EMP-0318,2026-06-01,0.998,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00
4,AN-0005,AF-03112,EMP-0347,2026-09-01,0.992,Y,NO_ATTENDANCE,No presence recorded for the whole month,NaN,iforest-attendance-0.1,2026-10-01 02:00:00


In [161]:
attendance_sample["model_version"].value_counts(dropna=False)

model_version
iforest-attendance-0.1    221
Name: count, dtype: int64

In [162]:
attendance_sample["money_at_risk"].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: money_at_risk, dtype: float64

In [165]:
attendance_sample.isna().sum()

result_id           0
source_dataset      0
record_id           0
employee_id         0
period              0
anomaly_score       0
flagged             0
reason_code         0
reason_text         0
money_at_risk     221
model_version       0
scored_at           0
dtype: int64

In [164]:
api_path = "/workspace/08_api"
if api_path not in sys.path:
    sys.path.insert(0, api_path)
import app.schemas
importlib.reload(app.schemas)
from app.schemas import AnomalyResponse
from datetime import date, datetime
sample_response = AnomalyResponse(
    result_id="AN-TEST-0001",
    source_dataset="attendance_monthly_features",
    record_id="AF-TEST-001",
    employee_id="EMP-0001",
    period=date(2026, 10, 1),
    anomaly_score=0.75,
    flagged="Y",
    reason_code="FREQUENT_LATE",
    reason_text="15 late arrivals this month",
    money_at_risk=None,
    model_version="iforest-attendance-0.1",
    scored_at=datetime.now()
)
print("Attendance response validation: PASSED")
print("Money at risk:", sample_response.money_at_risk)
print("Model version:", sample_response.model_version)
print("Response fields:", len(sample_response.model_dump()))

Attendance response validation: PASSED
Money at risk: None
Model version: iforest-attendance-0.1
Response fields: 12


In [166]:
main_code = '''
from contextlib import asynccontextmanager
from pathlib import Path
from datetime import datetime
from uuid import uuid4

import joblib
import pandas as pd

from fastapi import FastAPI, Request
from .schemas import AttendanceRequest, AnomalyResponse


PROJECT_ROOT = Path(__file__).resolve().parents[2]
MODELS_DIR = PROJECT_ROOT / "models"


@asynccontextmanager
async def lifespan(app: FastAPI):

    attendance_path = MODELS_DIR / "attendance_detector.joblib"
    payroll_path = MODELS_DIR / "payroll_detector.joblib"
    reason_codes_path = MODELS_DIR / "reason_codes.joblib"

    required_files = [
        attendance_path,
        payroll_path,
        reason_codes_path
    ]

    for file in required_files:
        if not file.exists():
            raise FileNotFoundError(
                f"Required model artifact not found: {file}"
            )

    app.state.artifacts = {
        "attendance": joblib.load(attendance_path),
        "payroll": joblib.load(payroll_path),
        "reason_codes": joblib.load(reason_codes_path)
    }

    print("All anomaly service artifacts loaded successfully.")

    yield

    app.state.artifacts = {}


app = FastAPI(
    title="CortexHR Anomaly Service",
    description="Attendance and payroll anomaly detection API",
    version="1.0.0",
    lifespan=lifespan
)


@app.get("/health")
def health_check(request: Request):

    return {
        "status": "healthy",
        "loaded_artifacts": list(
            request.app.state.artifacts.keys()
        )
    }


@app.post(
    "/v1/anomaly/attendance",
    response_model=AnomalyResponse
)
def score_attendance(
    payload: AttendanceRequest,
    request: Request
):

    artifact = request.app.state.artifacts["attendance"]

    model = artifact["model"]

    # Prepare model input using exact training feature order
    features = pd.DataFrame(
        [
            {
                column: getattr(payload, column)
                for column in artifact["feature_columns"]
            }
        ],
        columns=artifact["feature_columns"]
    )

    # Apply saved median values if any feature is missing
    medians = artifact["median_values"]

    features = features.fillna(medians)

    # Isolation Forest prediction
    prediction = int(model.predict(features)[0])

    model_anomaly = prediction == -1

    # Anomaly score: higher means more anomalous
    raw_score = float(-model.decision_function(features)[0])

    score_min = artifact["score_min"]
    score_max = artifact["score_max"]

    if score_max == score_min:
        anomaly_score = 0.0
    else:
        anomaly_score = (
            (raw_score - score_min) / (score_max - score_min)
        )

    anomaly_score = max(0.0, min(1.0, anomaly_score))

    # Apply Task 5 business rules
    attendance_rate = payload.attendance_rate

    zero_attendance = attendance_rate == 0

    invalid_attendance = not (
        0 <= attendance_rate <= 1
    )

    rule_anomaly = zero_attendance or invalid_attendance

    flagged = model_anomaly or rule_anomaly

    # Apply validated Task 7 reason-code rules
    reason_rules = (
        request.app.state.artifacts["reason_codes"]["attendance"]["rules"]
    )

    if not flagged:
        reason_code = "NO_ANOMALY"
        reason_text = "No attendance anomaly detected."

    elif (
        attendance_rate
        == reason_rules["NO_ATTENDANCE"]["value"]
    ):
        reason_code = "NO_ATTENDANCE"
        reason_text = "No presence recorded for the whole month"

    elif (
        payload.longest_unpaid_streak_days
        >= reason_rules["LONG_UNPAID_ABSENCE"]["value"]
    ):
        reason_code = "LONG_UNPAID_ABSENCE"
        reason_text = (
            f"{payload.longest_unpaid_streak_days} "
            "consecutive unpaid absence days"
        )

    elif (
        payload.late_arrivals
        >= reason_rules["FREQUENT_LATE"]["value"]
    ):
        reason_code = "FREQUENT_LATE"
        reason_text = (
            f"{payload.late_arrivals} late arrivals this month"
        )

    else:
        reason_code = "PATTERN_SHIFT"
        reason_text = (
            "Attendance pattern differs sharply from peers "
            "and own history"
        )

    return {
        "result_id": f"AN-{uuid4().hex[:8].upper()}",
        "source_dataset": "attendance_monthly_features",
        "record_id": payload.attendance_feature_id,
        "employee_id": payload.employee_id,
        "period": payload.month,
        "anomaly_score": anomaly_score,
        "flagged": "Y" if flagged else "N",
        "reason_code": reason_code,
        "reason_text": reason_text,
        "money_at_risk": None,
        "model_version": "iforest-attendance-0.1",
        "scored_at": datetime.now()
    }
'''

main_path = Path("/workspace/08_api/app/main.py")

main_path.write_text(
    main_code.strip() + "\n",
    encoding="utf-8"
)


Attendance endpoint implemented!
Python syntax: VALID
Endpoint: POST /v1/anomaly/attendance


In [ ]:
py_compile.compile(str(main_path), doraise=True)

In [ ]:
print("Attendance endpoint implemented!")
print("Python syntax: VALID")
print("Endpoint: POST /v1/anomaly/attendance")

In [ ]:
# before we tested it with startup now in real world

In [170]:
api_path = "/workspace/08_api"
if api_path not in sys.path:
    sys.path.insert(0, api_path)
import app.schemas
import app.main
importlib.reload(app.schemas)
importlib.reload(app.main)
from app.schemas import AttendanceRequest
from fastapi.testclient import TestClient
from app.main import app

# Load one real attendance record
attendance_df = pd.read_excel(
    "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx",
    sheet_name="attendance_monthly_features"
)
# Prepare request fields
attendance_fields = list(AttendanceRequest.model_fields)

row = attendance_df.dropna(
    subset=attendance_fields
).iloc[0]
payload = {
    field: row[field]
    for field in attendance_fields
}
# Validate and convert to JSON-compatible data
payload = AttendanceRequest(
    **payload
).model_dump(mode="json")
# Send request to API
with TestClient(app) as client:

    response = client.post(
        "/v1/anomaly/attendance",
        json=payload
    )
    print("HTTP Status:", response.status_code)
    print("\nAPI Response:")
    print(response.json())
    assert response.status_code == 200
    assert len(response.json()) == 12
    print("\nAttendance endpoint test passed!")

All anomaly service artifacts loaded successfully.
HTTP Status: 200

API Response:
{'result_id': 'AN-8099EDE1', 'source_dataset': 'attendance_monthly_features', 'record_id': 'AF-00001', 'employee_id': 'EMP-0002', 'period': '2025-04-01', 'anomaly_score': 0.09083938685895167, 'flagged': 'N', 'reason_code': 'NO_ANOMALY', 'reason_text': 'No attendance anomaly detected.', 'money_at_risk': None, 'model_version': 'iforest-attendance-0.1', 'scored_at': '2026-10-31T09:21:15.140311'}

Attendance endpoint test passed!


In [171]:
#Test invalid attendance input (HTTP 422)

In [172]:
from fastapi.testclient import TestClient
import copy
invalid_payload = copy.deepcopy(payload)
invalid_payload.pop("attendance_rate")
with TestClient(app) as client:
    response = client.post(
        "/v1/anomaly/attendance",
        json=invalid_payload
    )
    print("HTTP Status:", response.status_code)
    print("\nValidation response:")
    print(response.json())
    assert response.status_code == 422
print("\nInvalid attendance input test passed!")

All anomaly service artifacts loaded successfully.
HTTP Status: 422

Validation response:
{'detail': [{'type': 'missing', 'loc': ['body', 'attendance_rate'], 'msg': 'Field required', 'input': {'attendance_feature_id': 'AF-00001', 'employee_id': 'EMP-0002', 'month': '2025-04-01', 'job_level': 4, 'scheduled_days': 20, 'present_days': 20, 'wfh_days': 18, 'half_days_unpaid': 0, 'paid_leave_days': 0, 'unpaid_absence_days': 0, 'longest_unpaid_streak_days': 0, 'late_arrivals': 1, 'avg_hours_worked': 8.5, 'overtime_hours': 0.0}}]}

Invalid attendance input test passed!


In [175]:
import importlib
import app.schemas

importlib.reload(app.schemas)

from app.schemas import PayrollRequest

print("Duplicate flag:",
      PayrollRequest.model_fields["matching_payroll_record"].default)

print("Duplicate partner:",
      PayrollRequest.model_fields["duplicate_partner_id"].default)

Duplicate flag: False
Duplicate partner: None


In [176]:
#add payroll endpoint 

In [179]:
main_path = Path("/workspace/08_api/app/main.py")
main_source = main_path.read_text(encoding="utf-8")

# Update imports
if "import numpy as np" not in main_source:
    main_source = main_source.replace(
        "import pandas as pd",
        "import pandas as pd\nimport numpy as np"
    )

main_source = main_source.replace(
    "from .schemas import AttendanceRequest, AnomalyResponse",
    "from .schemas import AttendanceRequest, PayrollRequest, AnomalyResponse"
)

payroll_endpoint = '''
@app.post(
    "/v1/anomaly/payroll",
    response_model=AnomalyResponse
)
def score_payroll(
    payload: PayrollRequest,
    request: Request
):

    artifact = request.app.state.artifacts["payroll"]

    model = artifact["model"]
    imputer = artifact["imputer"]
    feature_columns = artifact["feature_columns"]

    # 1. Calculate the three training ratios
    ratio_values = {
        "gross_fixed_ratio": (
            payload.gross_pay / payload.monthly_fixed_gross
            if payload.monthly_fixed_gross != 0
            else np.nan
        ),
        "deduction_ratio": (
            payload.total_deductions / payload.gross_pay
            if payload.gross_pay != 0
            else np.nan
        ),
        "net_ratio": (
            payload.net_pay / payload.gross_pay
            if payload.gross_pay != 0
            else np.nan
        )
    }

    X = pd.DataFrame(
        [[ratio_values[column] for column in feature_columns]],
        columns=feature_columns
    )

    # 2. Apply the original fitted imputer
    X_imputed = imputer.transform(X)

    # 3. Predict using the original fitted model
    prediction = int(model.predict(X_imputed)[0])
    model_anomaly = prediction == -1

    raw_score = float(model.decision_function(X_imputed)[0])

    # Use the Task 6 training score range
    score_min = artifact["score_min"]
    score_max = artifact["score_max"]

    if score_max == score_min:
        anomaly_score = 0.0
    else:
        anomaly_score = (
            (score_max - raw_score) / (score_max - score_min)
        )

    anomaly_score = max(0.0, min(1.0, anomaly_score))

    # 4. Apply Task 6 business rules
    negative_net_pay = payload.net_pay < 0

    zero_net_pay = (
        payload.net_pay == 0
        and payload.payable_days > 0
    )

    duplicate_payment = payload.matching_payroll_record

    flagged = (
        model_anomaly
        or negative_net_pay
        or zero_net_pay
        or duplicate_payment
    )

    # 5. Calculate prorated expected fixed pay
    calendar_days = pd.Timestamp(payload.month).days_in_month

    expected_fixed_pay = (
        payload.monthly_fixed_gross
        * payload.payable_days
        / calendar_days
    )

    gross = payload.gross_pay
    net = payload.net_pay

    money_at_risk = 0.0

    # 6. Apply Task 7 reason-code priority
    if not flagged:

        reason_code = "NO_ANOMALY"
        reason_text = "No payroll anomaly detected."

    elif negative_net_pay:

        reason_code = "NEGATIVE_NET_PAY"
        money_at_risk = abs(net)

        reason_text = (
            f"Deductions exceed gross pay by ₹{money_at_risk:,.2f}, "
            "resulting in negative net pay."
        )

    elif zero_net_pay:

        reason_code = "ZERO_PAY"
        money_at_risk = max(expected_fixed_pay - gross, 0)

        reason_text = (
            f"No net pay was recorded despite expected "
            f"prorated fixed pay of ₹{money_at_risk:,.2f}."
        )

    elif duplicate_payment:

        reason_code = "DOUBLE_PAYMENT"

        partner = payload.duplicate_partner_id or "matching record"

        is_additional = payload.payroll_item_id.startswith("PI-D")

        if is_additional:
            money_at_risk = max(net, 0)

            reason_text = (
                f"Additional record matches original payment {partner}; "
                f"verify whether ₹{money_at_risk:,.2f} was paid twice."
            )
        else:
            money_at_risk = 0.0

            reason_text = (
                f"This payment matches additional record {partner}; "
                "financial exposure is assigned to that record "
                "to avoid double-counting."
            )

    else:

        reason_code = "STATISTICAL_OUTLIER"

        money_at_risk = abs(gross - expected_fixed_pay)

        reason_text = (
            f"Gross pay differs from prorated fixed pay by "
            f"₹{money_at_risk:,.2f}; review whether overtime, variable "
            "pay or a salary change explains the difference."
        )

    # 7. Return official response format
    return {
        "result_id": f"AN-{uuid4().hex[:8].upper()}",
        "source_dataset": "payroll_monthly_features",
        "record_id": payload.payroll_item_id,
        "employee_id": payload.employee_id,
        "period": payload.month,
        "anomaly_score": anomaly_score,
        "flagged": "Y" if flagged else "N",
        "reason_code": reason_code,
        "reason_text": reason_text,
        "money_at_risk": float(money_at_risk),
        "model_version": "iforest-payroll-0.1+rules",
        "scored_at": datetime.now()
    }
'''

if '"/v1/anomaly/payroll"' in main_source:
    print("Payroll endpoint already exists. No duplicate added.")
else:
    main_source = (main_source.rstrip()+ "\n\n"+ payroll_endpoint.strip()+ "\n")
main_path.write_text(main_source, encoding="utf-8")

Payroll endpoint already exists. No duplicate added.


9384

In [180]:
py_compile.compile(str(main_path), doraise=True)

print("Payroll endpoint implemented successfully!")
print("Python syntax: VALID")
print("Endpoint: POST /v1/anomaly/payroll")

Payroll endpoint implemented successfully!
Python syntax: VALID
Endpoint: POST /v1/anomaly/payroll


In [ ]:
# testing payroll endpoint

In [181]:
api_path = "/workspace/08_api"
if api_path not in sys.path:
    sys.path.insert(0, api_path)
import app.schemas
import app.main
importlib.reload(app.schemas)
importlib.reload(app.main)
from app.schemas import PayrollRequest
from app.main import app
from fastapi.testclient import TestClient

# Load a real payroll record
payroll_df = pd.read_excel(
    "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx",
    sheet_name="payroll_monthly_features"
)

row = payroll_df.iloc[0]

payload = {
    "payroll_item_id": row["payroll_item_id"],
    "employee_id": row["employee_id"],
    "run_id": row["run_id"],
    "month": pd.Timestamp(row["month"]).date().isoformat(),
    "monthly_fixed_gross": float(row["monthly_fixed_gross"]),
    "payable_days": float(row["payable_days"]),
    "gross_pay": float(row["gross_pay"]),
    "total_deductions": float(row["total_deductions"]),
    "net_pay": float(row["net_pay"]),
    "matching_payroll_record": False,
    "duplicate_partner_id": None
}

# Validate input
payload = PayrollRequest(**payload).model_dump(mode="json")

# Test API
with TestClient(app) as client:

    response = client.post(
        "/v1/anomaly/payroll",
        json=payload
    )
    print("HTTP Status:", response.status_code)
    print("\nPayroll API response:")
    print(response.json())
    assert response.status_code == 200
    assert len(response.json()) == 12
print("\nPayroll endpoint test passed!")

All anomaly service artifacts loaded successfully.
HTTP Status: 200

Payroll API response:
{'result_id': 'AN-467563E0', 'source_dataset': 'payroll_monthly_features', 'record_id': 'PI-000001', 'employee_id': 'EMP-0002', 'period': '2025-04-01', 'anomaly_score': 0.319132234870734, 'flagged': 'N', 'reason_code': 'NO_ANOMALY', 'reason_text': 'No payroll anomaly detected.', 'money_at_risk': 0.0, 'model_version': 'iforest-payroll-0.1+rules', 'scored_at': '2026-10-31T09:36:00.700795'}

Payroll endpoint test passed!


In [ ]:
#creating API test file 

In [185]:
tests_path = Path("/workspace/tests")
tests_path.mkdir(exist_ok=True)
test_code = r'''
from pathlib import Path
import sys

import pytest
from fastapi.testclient import TestClient

# Make the API application importable
API_DIR = Path(__file__).resolve().parents[1] / "08_api"
sys.path.insert(0, str(API_DIR))

from app.main import app


@pytest.fixture(scope="module")
def client():
    with TestClient(app) as test_client:
        yield test_client


def attendance_payload():
    return {
        "attendance_feature_id": "AF-TEST-001",
        "employee_id": "EMP-TEST-001",
        "month": "2026-10-01",
        "job_level": 4,
        "scheduled_days": 20,
        "present_days": 18,
        "wfh_days": 0,
        "half_days_unpaid": 0,
        "paid_leave_days": 0,
        "unpaid_absence_days": 2,
        "longest_unpaid_streak_days": 2,
        "late_arrivals": 1,
        "avg_hours_worked": 8.0,
        "overtime_hours": 0.0,
        "attendance_rate": 0.9
    }


def payroll_payload():
    return {
        "payroll_item_id": "PI-TEST-001",
        "employee_id": "EMP-TEST-001",
        "run_id": "RUN-TEST-001",
        "month": "2026-10-01",
        "monthly_fixed_gross": 50000.0,
        "payable_days": 30.0,
        "gross_pay": 50000.0,
        "total_deductions": 5000.0,
        "net_pay": 45000.0,
        "matching_payroll_record": False,
        "duplicate_partner_id": None
    }


OFFICIAL_FIELDS = {
    "result_id",
    "source_dataset",
    "record_id",
    "employee_id",
    "period",
    "anomaly_score",
    "flagged",
    "reason_code",
    "reason_text",
    "money_at_risk",
    "model_version",
    "scored_at"
}


def test_health_check(client):
    response = client.get("/health")

    assert response.status_code == 200

    assert set(response.json()["loaded_artifacts"]) == {
        "attendance",
        "payroll",
        "reason_codes"
    }


def test_attendance_valid_request(client):
    response = client.post(
        "/v1/anomaly/attendance",
        json=attendance_payload()
    )

    assert response.status_code == 200
    assert set(response.json()) == OFFICIAL_FIELDS
    assert response.json()["source_dataset"] == (
        "attendance_monthly_features"
    )


def test_attendance_missing_field_returns_422(client):
    payload = attendance_payload()
    payload.pop("attendance_rate")

    response = client.post(
        "/v1/anomaly/attendance",
        json=payload
    )

    assert response.status_code == 422


def test_payroll_valid_request(client):
    response = client.post(
        "/v1/anomaly/payroll",
        json=payroll_payload()
    )

    assert response.status_code == 200
    assert set(response.json()) == OFFICIAL_FIELDS
    assert response.json()["source_dataset"] == (
        "payroll_monthly_features"
    )


def test_payroll_missing_field_returns_422(client):
    payload = payroll_payload()
    payload.pop("net_pay")

    response = client.post(
        "/v1/anomaly/payroll",
        json=payload
    )

    assert response.status_code == 422


def test_payroll_duplicate_detection(client):
    payload = payroll_payload()

    payload["payroll_item_id"] = "PI-D-TEST-001"
    payload["matching_payroll_record"] = True
    payload["duplicate_partner_id"] = "PI-ORIGINAL-001"

    response = client.post(
        "/v1/anomaly/payroll",
        json=payload
    )

    assert response.status_code == 200
    assert response.json()["reason_code"] == "DOUBLE_PAYMENT"
    assert response.json()["money_at_risk"] == 45000.0
'''

test_file = tests_path / "test_api.py"

test_file.write_text(
    test_code.strip() + "\n",
    encoding="utf-8"
)

print("API test file created successfully!")
print("Location:", test_file)
print("Total tests:", test_code.count("def test_"))
print("File exists:", test_file.exists())

API test file created successfully!
Location: /workspace/tests/test_api.py
Total tests: 6
File exists: True


In [186]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "/workspace/tests/test_api.py",
        "-v"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

print("Pytest exit code:", result.returncode)

============================= test session starts ==============================
platform linux -- Python 3.12.15, pytest-9.2.0, pluggy-1.7.0 -- /usr/local/bin/python3.12
cachedir: .pytest_cache
rootdir: /workspace
plugins: anyio-4.15.1, platformdirs-4.13.0
collecting ... collected 6 items

tests/test_api.py::test_health_check PASSED                              [ 16%]
tests/test_api.py::test_attendance_valid_request PASSED                  [ 33%]
tests/test_api.py::test_attendance_missing_field_returns_422 PASSED      [ 50%]
tests/test_api.py::test_payroll_valid_request PASSED                     [ 66%]
tests/test_api.py::test_payroll_missing_field_returns_422 PASSED         [ 83%]
tests/test_api.py::test_payroll_duplicate_detection PASSED               [100%]

=============================== warnings summary ===============================
../usr/local/lib/python3.12/site-packages/fastapi/testclient.py:1
  /usr/local/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDepr

In [194]:
# save and documenting 

In [196]:
readme_path = Path("/workspace/08_api/README.md")

readme_content = r'''
# CortexHR Anomaly Detection API

## 1. Overview

FastAPI service for attendance and payroll anomaly detection.

## 2. API Endpoints

| Method | Endpoint | Description |
|---|---|---|
| GET | /health | Health check |
| POST | /v1/anomaly/attendance | Attendance anomaly detection |
| POST | /v1/anomaly/payroll | Payroll anomaly detection |

## 3. Model Artifacts

The service loads these artifacts once at startup:

- models/attendance_detector.joblib
- models/payroll_detector.joblib
- models/reason_codes.joblib

## 4. Install Dependencies

From the project root:

    pip install -r requirements.txt

## 5. Run the API

    cd /workspace/08_api
    python -m uvicorn app.main:app --host 0.0.0.0 --port 8000

## 6. Test Health Endpoint

    curl -i http://127.0.0.1:8000/health

Verified: HTTP 200 OK.

## 7. Test Attendance Endpoint

    curl -i -X POST http://127.0.0.1:8000/v1/anomaly/attendance \
    -H "Content-Type: application/json" \
    -d '{"attendance_feature_id":"AF-00001","employee_id":"EMP-0002","month":"2025-04-01","job_level":4,"scheduled_days":20,"present_days":20,"wfh_days":18,"half_days_unpaid":0,"paid_leave_days":0,"unpaid_absence_days":0,"longest_unpaid_streak_days":0,"late_arrivals":1,"avg_hours_worked":8.5,"overtime_hours":0,"attendance_rate":1.0}'

Verified: HTTP 200 OK.

## 8. Test Payroll Endpoint

    curl -i -X POST http://127.0.0.1:8000/v1/anomaly/payroll \
    -H "Content-Type: application/json" \
    -d '{"payroll_item_id":"PI-TEST-001","employee_id":"EMP-TEST-001","run_id":"RUN-TEST-001","month":"2026-10-01","monthly_fixed_gross":50000,"payable_days":30,"gross_pay":50000,"total_deductions":5000,"net_pay":45000,"matching_payroll_record":false,"duplicate_partner_id":null}'

Verified: HTTP 200 OK.

## 9. Test Invalid Input

A payroll request missing net_pay was submitted.

Verified: HTTP 422 Unprocessable Entity.

## 10. Automated Tests

Run:

    python -m pytest /workspace/tests/test_api.py -v

Verified result:

    6 passed, 1 warning

Tests cover:
1. Health check.
2. Valid attendance request.
3. Invalid attendance request.
4. Valid payroll request.
5. Invalid payroll request.
6. Duplicate payroll detection.

## 11. Interactive API Documentation

While the service is running:

    http://127.0.0.1:8000/docs
'''

readme_path.write_text(
    readme_content.strip() + "\n",
    encoding="utf-8"
)

print("README created successfully!")
print("Location:", readme_path)
print("Size:", readme_path.stat().st_size, "bytes")
print("Exists:", readme_path.exists())

README created successfully!
Location: /workspace/08_api/README.md
Size: 2269 bytes
Exists: True


In [198]:
# Create batch request schema

In [202]:
importlib.reload(app.schemas)
from app.schemas import (AttendanceBatchRequest,PayrollBatchRequest)
schemas = [AttendanceBatchRequest,PayrollBatchRequest]
for schema in schemas:
    details = schema.model_json_schema()["properties"]["records"]
    print("\nSchema:", schema.__name__)
    print("Minimum records:", details.get("minItems"))
    print("Maximum records:", details.get("maxItems"))
    assert details.get("minItems") == 1
    assert details.get("maxItems") == 100
print("\nBoth batch schemas verified successfully!")


Schema: AttendanceBatchRequest
Minimum records: 1
Maximum records: 100

Schema: PayrollBatchRequest
Minimum records: 1
Maximum records: 100

Both batch schemas verified successfully!


In [203]:
#Adding one payroll batch endpoint

In [206]:
main_path = Path("/workspace/08_api/app/main.py")
source = main_path.read_text(encoding="utf-8")
print("REGISTERED ENDPOINTS\n")
for line in source.splitlines():
    if line.strip().startswith("@app.") or "/v1/anomaly/" in line:
        print(line.strip())

REGISTERED ENDPOINTS

@app.get("/health")
@app.post(
"/v1/anomaly/attendance",
@app.post(
"/v1/anomaly/payroll",
@app.post(
"/v1/anomaly/payroll/batch",
